# Yelp Query-Based Content Recommendation Model — EMR Automation Edition

This notebook is the EMR/S3 deployment version of **`Yelp_Content_Based_Recommender_Final.ipynb`**.

### Model logic retained
- Category token creation and binary `CountVectorizer` representation.
- L2 normalization of category vectors.
- Query-based category, location, attribute, rating, and review-confidence scoring.
- Train/validation/test split of offline evaluation queries.
- Compact weight grid search and final test evaluation.
- Top-K business recommendation output.

### EMR-only changes
- Replaced Databricks `/Volumes/...` paths with S3 paths.
- Added automatic input-path discovery and schema preflight checks.
- Added EMR-safe Spark configuration for the available cluster.
- Replaced local `open()` JSON writes with distributed S3-safe writes.
- Added versioned model directories, `BUILDING`/`READY` manifest states, reload validation, and a saved-model smoke test.
- No Databricks utilities, widgets, `%pip`, `%run`, or `display()` calls are used.

> **Cluster target:** EMR Studio PySpark kernel, `m4.large` primary and `m5.4xlarge` core node(s).
>
> Run the notebook from top to bottom. Normally, only `S3_BUCKET`, `MODEL_VERSION`, and possibly `BUSINESS_INPUT_CANDIDATES` need editing.


In [1]:
%%configure -f
{
    "conf": {
        "spark.pyspark.python": "python3",
        "spark.pyspark.virtualenv.enabled": "true",
        "spark.pyspark.virtualenv.type": "native",
        "spark.pyspark.virtualenv.bin.path": "/usr/bin/virtualenv"
    }
}

In [2]:
# ============================================================
# EMR DEPENDENCY FIX: INSTALL NUMPY
# ============================================================

import sys

python_version = sys.version_info

if python_version >= (3, 12):
    NUMPY_PACKAGE = "numpy==1.26.4"
elif python_version >= (3, 8):
    NUMPY_PACKAGE = "numpy==1.24.4"
else:
    NUMPY_PACKAGE = "numpy==1.21.6"

print("Python version:", sys.version)
print("Installing:", NUMPY_PACKAGE)

sc.install_pypi_package(
    NUMPY_PACKAGE,
    "https://pypi.org/simple"
)

print("NumPy installation completed.")

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
1,application_1785604648071_0002,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Python version: 3.9.25 (main, May 25 2026, 00:00:00) 
[GCC 11.5.0 20240719 (Red Hat 11.5.0-5)]
Installing: numpy==1.24.4

NumPy installation completed.

In [3]:
# ============================================================
# CELL 0.3: VERIFY NUMPY AND PYSPARK ML
# ============================================================

import numpy as np

from pyspark.ml.feature import (
    CountVectorizer,
    CountVectorizerModel,
    Normalizer,
)

from pyspark.ml.linalg import VectorUDT

print("NumPy version:", np.__version__)
print("CountVectorizer imported successfully.")
print("Normalizer imported successfully.")
print("VectorUDT imported successfully.")
print("Dependency verification passed.")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

NumPy version: 1.24.4
CountVectorizer imported successfully.
Normalizer imported successfully.
VectorUDT imported successfully.
Dependency verification passed.

In [4]:
# ============================================================
# CELL 1: IMPORTS AND SPARK SESSION
# ============================================================

import json
import math
import sys
import numpy as np

from datetime import datetime, timezone

from pyspark.sql import SparkSession, Row
from pyspark.sql import functions as F
from pyspark.sql.window import Window

from pyspark.sql.types import (
    ArrayType,
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)

from pyspark.ml.feature import (
    CountVectorizer,
    CountVectorizerModel,
    Normalizer,
)

from pyspark.ml.linalg import VectorUDT

spark = SparkSession.builder.getOrCreate()

print("Spark session ready.")
print("Spark version:", spark.version)
print("Application ID:", spark.sparkContext.applicationId)
print("Python version:", sys.version)
print("NumPy version:", np.__version__)
print("All imports completed successfully.")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Spark session ready.
Spark version: 3.5.3-amzn-0
Application ID: application_1785604648071_0002
Python version: 3.9.25 (main, May 25 2026, 00:00:00) 
[GCC 11.5.0 20240719 (Red Hat 11.5.0-5)]
NumPy version: 1.24.4
All imports completed successfully.

In [2]:
# ============================================================
# CELL 2: EMR RUNTIME CONFIGURATION
# ============================================================
# These settings affect execution efficiency only; they do not
# change the recommendation model or its scoring logic.

DEFAULT_PARALLELISM = max(1, spark.sparkContext.defaultParallelism)
SHUFFLE_PARTITIONS = max(64, min(400, DEFAULT_PARALLELISM * 3))
OUTPUT_PARTITIONS = max(8, min(64, DEFAULT_PARALLELISM))

spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", str(SHUFFLE_PARTITIONS))

spark.sparkContext.setLogLevel("WARN")

print("Default parallelism:", DEFAULT_PARALLELISM)
print("Shuffle partitions:", SHUFFLE_PARTITIONS)
print("Output partitions:", OUTPUT_PARTITIONS)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Default parallelism: 8
Shuffle partitions: 64
Output partitions: 8

In [5]:
# ============================================================
# CELL 3: S3 INPUT AND VERSIONED OUTPUT CONFIGURATION
# ============================================================

S3_BUCKET = "yelpdataset-project"
MODEL_VERSION = "v1"

# The notebook checks these paths in order and uses the first
# existing Parquet location. Keep the correct location first.
BUSINESS_INPUT_CANDIDATES = [
    f"s3://{S3_BUCKET}/gold_layer/ml/content_based_filtering/content.parquet",
    f"s3://{S3_BUCKET}/gold_layer/ml/content_based_filtering/content/",
    f"s3://{S3_BUCKET}/gold_layer/ml/content_based_filtering/",
    f"s3://{S3_BUCKET}/gold_layer/ml/content_based_data/",
]

CONTENT_OUTPUT_PATH = (
    f"s3://{S3_BUCKET}/gold_layer/ml/"
    f"content_based_recommender_model/{MODEL_VERSION}/"
)

BUSINESS_VECTOR_OUTPUT_PATH = CONTENT_OUTPUT_PATH + "business_feature_vectors/"
VALIDATION_RESULTS_OUTPUT_PATH = CONTENT_OUTPUT_PATH + "evaluation/validation_results/"
TEST_METRICS_OUTPUT_PATH = CONTENT_OUTPUT_PATH + "evaluation/test_metrics/"
RECOMMENDATIONS_OUTPUT_PATH = CONTENT_OUTPUT_PATH + "sample_recommendations/"
MODEL_CONFIG_OUTPUT_PATH = CONTENT_OUTPUT_PATH + "model_configuration/"
MODEL_MANIFEST_OUTPUT_PATH = CONTENT_OUTPUT_PATH + "model_manifest/"

MODEL_ARTIFACTS_PATH = CONTENT_OUTPUT_PATH + "model_artifacts/"
CATEGORY_VECTORIZER_MODEL_PATH = MODEL_ARTIFACTS_PATH + "category_vectorizer_model/"
NORMALIZER_MODEL_PATH = MODEL_ARTIFACTS_PATH + "normalizer/"

print("Model version:", MODEL_VERSION)
print("Output root:", CONTENT_OUTPUT_PATH)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Model version: v1
Output root: s3://yelpdataset-project/gold_layer/ml/content_based_recommender_model/v1/

In [7]:
# ============================================================
# CELL 4: HADOOP/S3 PATH UTILITIES
# ============================================================

def path_exists(path):
    """Return True when an S3/HDFS path exists for this Spark session."""
    j_path = spark._jvm.org.apache.hadoop.fs.Path(path)
    file_system = j_path.getFileSystem(spark._jsc.hadoopConfiguration())
    return bool(file_system.exists(j_path))


def delete_path_if_exists(path):
    """Delete an S3/HDFS path recursively when it already exists."""
    j_path = spark._jvm.org.apache.hadoop.fs.Path(path)
    file_system = j_path.getFileSystem(spark._jsc.hadoopConfiguration())
    if file_system.exists(j_path):
        file_system.delete(j_path, True)


def write_json_document(document, output_path):
    """Write one JSON document to S3 without using local Python open()."""
    payload = json.dumps(document, sort_keys=True)
    (
        spark.createDataFrame([(payload,)], ["value"])
        .coalesce(1)
        .write.mode("overwrite")
        .text(output_path)
    )


def read_json_document(input_path):
    """Read a JSON document written by write_json_document()."""
    row = spark.read.text(input_path).first()
    if row is None or row["value"] is None:
        raise ValueError(f"No JSON document found at {input_path}")
    return json.loads(row["value"])

print("S3 path utilities ready.")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

S3 path utilities ready.

In [8]:
# ============================================================
# CELL 5: RESOLVE THE BUSINESS INPUT PATH
# ============================================================

BUSINESS_FEATURES_PATH = None

for candidate_path in BUSINESS_INPUT_CANDIDATES:
    try:
        if path_exists(candidate_path):
            BUSINESS_FEATURES_PATH = candidate_path
            break
    except Exception as path_error:
        print(f"Could not inspect {candidate_path}: {path_error}")

if BUSINESS_FEATURES_PATH is None:
    raise FileNotFoundError(
        "None of the configured business input paths exists. "
        "Update BUSINESS_INPUT_CANDIDATES in Cell 3."
    )

print("Resolved business input path:", BUSINESS_FEATURES_PATH)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Resolved business input path: s3://yelpdataset-project/gold_layer/ml/content_based_filtering/

In [9]:
# ============================================================
# CELL 6: MARK THIS MODEL VERSION AS BUILDING
# ============================================================
# The READY manifest is written only after all artifacts pass
# the reload and consistency checks.

BUILD_STARTED_AT_UTC = datetime.now(timezone.utc).isoformat()

building_manifest = {
    "model_name": "Yelp_Content_Based_Recommender",
    "model_version": MODEL_VERSION,
    "status": "BUILDING",
    "build_started_at_utc": BUILD_STARTED_AT_UTC,
    "input_path": BUSINESS_FEATURES_PATH,
    "output_root": CONTENT_OUTPUT_PATH,
}

write_json_document(building_manifest, MODEL_MANIFEST_OUTPUT_PATH)
print("BUILDING manifest saved.")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

BUILDING manifest saved.

In [10]:
# ============================================================
# CELL 7: LOAD BUSINESS FEATURES
# ============================================================

try:
    business_raw_df = spark.read.parquet(BUSINESS_FEATURES_PATH)
except Exception as error:
    raise RuntimeError(
        f"Unable to read Parquet business data from {BUSINESS_FEATURES_PATH}"
    ) from error

print("Input columns:")
print(business_raw_df.columns)

business_raw_df.printSchema()


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Input columns:
['business_id', 'categories', 'city', 'state', 'latitude', 'longitude', 'stars', 'review_count', 'is_open', 'attributes_restaurantspricerange2', 'attributes_wifi', 'attributes_outdoorseating']
root
 |-- business_id: string (nullable = true)
 |-- categories: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- stars: double (nullable = true)
 |-- review_count: integer (nullable = true)
 |-- is_open: integer (nullable = true)
 |-- attributes_restaurantspricerange2: string (nullable = true)
 |-- attributes_wifi: string (nullable = true)
 |-- attributes_outdoorseating: string (nullable = true)

In [11]:
# ============================================================
# CELL 8: CANONICALIZE KNOWN COLUMN NAMES
# ============================================================
# This allows the same notebook to tolerate common case/naming
# differences without changing feature meaning.

COLUMN_ALIASES = {
    "business_id": ["business_id", "businessid"],
    "name": ["name", "business_name"],
    "categories": ["categories", "category"],
    "city": ["city"],
    "state": ["state"],
    "latitude": ["latitude", "lat"],
    "longitude": ["longitude", "lon", "lng"],
    "stars": ["stars", "business_avg_rating", "avg_rating"],
    "review_count": ["review_count", "business_review_count"],
    "is_open": ["is_open", "open_status"],
    "attributes_restaurantspricerange2": [
        "attributes_restaurantspricerange2",
        "attributes_restaurants_price_range2",
        "restaurantspricerange2",
        "price_range",
    ],
    "attributes_wifi": ["attributes_wifi", "wifi", "has_wifi"],
    "attributes_outdoorseating": [
        "attributes_outdoorseating",
        "attributes_outdoor_seating",
        "outdoorseating",
        "outdoor_seating",
    ],
}


def canonicalize_columns(dataframe, aliases):
    result = dataframe
    current_lookup = {column_name.lower(): column_name for column_name in result.columns}

    for canonical_name, candidate_names in aliases.items():
        if canonical_name in result.columns:
            continue

        matched_name = None
        for candidate_name in candidate_names:
            if candidate_name.lower() in current_lookup:
                matched_name = current_lookup[candidate_name.lower()]
                break

        if matched_name is not None and matched_name != canonical_name:
            result = result.withColumnRenamed(matched_name, canonical_name)
            current_lookup[canonical_name.lower()] = canonical_name

    return result


business_df = canonicalize_columns(business_raw_df, COLUMN_ALIASES)
print("Canonicalized columns:")
print(business_df.columns)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Canonicalized columns:
['business_id', 'categories', 'city', 'state', 'latitude', 'longitude', 'stars', 'review_count', 'is_open', 'attributes_restaurantspricerange2', 'attributes_wifi', 'attributes_outdoorseating']

In [12]:
# ============================================================
# CELL 9: INPUT DATA PREFLIGHT VALIDATION
# ============================================================

REQUIRED_INPUT_COLUMNS = {
    "business_id",
    "categories",
    "city",
    "state",
    "stars",
    "review_count",
    "is_open",
    "attributes_restaurantspricerange2",
    "attributes_wifi",
    "attributes_outdoorseating",
}

missing_input_columns = sorted(REQUIRED_INPUT_COLUMNS - set(business_df.columns))

if missing_input_columns:
    raise ValueError(
        "Missing required input columns: " + ", ".join(missing_input_columns)
    )

# Add optional presentation columns when absent. This does not
# remove or change any source rows.
OPTIONAL_COLUMN_TYPES = {
    "name": "string",
    "latitude": "double",
    "longitude": "double",
}

for optional_column, data_type in OPTIONAL_COLUMN_TYPES.items():
    if optional_column not in business_df.columns:
        business_df = business_df.withColumn(
            optional_column, F.lit(None).cast(data_type)
        )

business_row_count = business_df.count()

if business_row_count == 0:
    raise ValueError("The business input dataset contains zero rows.")

null_business_id_count = business_df.filter(F.col("business_id").isNull()).count()
if null_business_id_count > 0:
    raise ValueError(f"Found {null_business_id_count} rows with null business_id.")

duplicate_business_id_count = (
    business_df.groupBy("business_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

if duplicate_business_id_count > 0:
    raise ValueError(
        f"Found {duplicate_business_id_count} duplicated business_id values."
    )

print("Input preflight passed.")
print("Business rows:", business_row_count)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Input preflight passed.
Business rows: 150346

In [13]:
# ============================================================
# CELL 10: BASIC INPUT INSPECTION
# ============================================================

business_df.select(
    "business_id",
    "name",
    "categories",
    "city",
    "state",
    "stars",
    "review_count",
    "is_open",
).show(5, truncate=False)

business_df.select([
    F.count(F.when(F.col(column_name).isNull(), column_name)).alias(column_name)
    for column_name in business_df.columns
]).show(truncate=False)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----------------------+----+-----------------------------------------------------------------------------------------------------+----------------+-----+-----+------------+-------+
|business_id           |name|categories                                                                                           |city            |state|stars|review_count|is_open|
+----------------------+----+-----------------------------------------------------------------------------------------------------+----------------+-----+-----+------------+-------+
|--l-mz2pV40R0lIL4BQOnw|NULL|Home Services, Shopping, Cabinetry, Home & Garden, Interior Design, Building Supplies, Kitchen & Bath|Reno            |NV   |4.0  |14          |1      |
|-0gRYq5UjMtZbELj0KHxzA|NULL|Food, Coffee & Tea                                                                                   |Bryn Mawr       |PA   |3.5  |128         |1      |
|-0yr__Vo4hOCZwvzRWd1zA|NULL|Men's Clothing, Shopping, Fashion                            

In [14]:
# ============================================================
# CELL 11: HANDLE MODEL-FEATURE NULLS
# ============================================================
# No source row is dropped here.

business_df = business_df.fillna({
    "categories": "Unknown",
    "city": "Unknown",
    "state": "Unknown",
    "attributes_restaurantspricerange2": "Unknown",
    "attributes_wifi": "Unknown",
    "attributes_outdoorseating": "Unknown",
})

business_df = (
    business_df
    .withColumn("stars", F.coalesce(F.col("stars").cast("double"), F.lit(0.0)))
    .withColumn(
        "review_count",
        F.greatest(
            F.coalesce(F.col("review_count").cast("long"), F.lit(0)),
            F.lit(0),
        ),
    )
)

print("Model-feature missing values handled without dropping rows.")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Model-feature missing values handled without dropping rows.

In [15]:
# ============================================================
# CELL 12: CREATE CLEAN CATEGORY TOKENS
# ============================================================

business_df = business_df.withColumn(
    "category_tokens",
    F.array_distinct(
        F.array_remove(
            F.transform(
                F.split(F.lower(F.col("categories")), ","),
                lambda category: F.trim(category),
            ),
            "",
        )
    ),
)

# Ensure every row has at least one token.
business_df = business_df.withColumn(
    "category_tokens",
    F.when(
        F.size(F.col("category_tokens")) > 0,
        F.col("category_tokens"),
    ).otherwise(F.array(F.lit("unknown"))),
)

business_df.select(
    "business_id", "categories", "category_tokens"
).show(10, truncate=False)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----------------------+-----------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------+
|business_id           |categories                                                                                           |category_tokens                                                                                        |
+----------------------+-----------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------+
|--l-mz2pV40R0lIL4BQOnw|Home Services, Shopping, Cabinetry, Home & Garden, Interior Design, Building Supplies, Kitchen & Bath|[home services, shopping, cabinetry, home & garden, interior design, building supplies, kitchen & bath]|
|-0gRYq5UjMtZbELj0KHxzA|Food, Coffee & Tea                                  

In [16]:
# ============================================================
# CELL 13: FIT CATEGORY COUNT VECTORIZER
# ============================================================

category_vectorizer = CountVectorizer(
    inputCol="category_tokens",
    outputCol="category_vector",
    binary=True,
    minDF=2.0,
)

category_vectorizer_model = category_vectorizer.fit(business_df)
business_df = category_vectorizer_model.transform(business_df)

if len(category_vectorizer_model.vocabulary) == 0:
    raise ValueError("The fitted category vocabulary is empty.")

print("Category vocabulary size:", len(category_vectorizer_model.vocabulary))

business_df.select(
    "business_id", "category_tokens", "category_vector"
).show(10, truncate=False)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Category vocabulary size: 1256
+----------------------+-------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------+
|business_id           |category_tokens                                                                                        |category_vector                                                         |
+----------------------+-------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------+
|--l-mz2pV40R0lIL4BQOnw|[home services, shopping, cabinetry, home & garden, interior design, building supplies, kitchen & bath]|(1256,[2,3,20,134,170,220,389],[1.0,1.0,1.0,1.0,1.0,1.0,1.0])           |
|-0gRYq5UjMtZbELj0KHxzA|[food, coffee & tea]                                                                                   |(1256,[1,15],[1.0,1.0])          

In [17]:
# ============================================================
# CELL 14: L2-NORMALIZE CATEGORY VECTORS
# ============================================================

normalizer = Normalizer(
    inputCol="category_vector",
    outputCol="category_vector_normalized",
    p=2.0,
)

business_df = normalizer.transform(business_df)

business_df.select(
    "business_id", "category_vector", "category_vector_normalized"
).show(10, truncate=False)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----------------------+------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|business_id           |category_vector                                                         |category_vector_normalized                                                                                                                                                                                     |
+----------------------+------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|--ARBQr1WMsTWiwOKOj-FQ|(1256,[5,8,107],[1.0,1.0,1.0])                            

In [18]:
# ============================================================
# CELL 15: CLEAN PRICE, WIFI, OUTDOOR SEATING, AND OPEN STATUS
# ============================================================

price_text = F.lower(
    F.trim(
        F.coalesce(
            F.col("attributes_restaurantspricerange2").cast("string"),
            F.lit("unknown"),
        )
    )
)

wifi_text = F.lower(
    F.trim(
        F.coalesce(F.col("attributes_wifi").cast("string"), F.lit("unknown"))
    )
)
wifi_text = F.regexp_replace(F.regexp_replace(wifi_text, "u'", ""), "'", "")

outdoor_text = F.lower(
    F.trim(
        F.coalesce(
            F.col("attributes_outdoorseating").cast("string"),
            F.lit("unknown"),
        )
    )
)
outdoor_text = F.regexp_replace(
    F.regexp_replace(outdoor_text, "u'", ""), "'", ""
)

open_text = F.lower(F.trim(F.col("is_open").cast("string")))

business_df = (
    business_df
    .withColumn(
        "price_range_numeric",
        F.when(
            F.regexp_extract(price_text, r"([1-4])", 1) != "",
            F.regexp_extract(price_text, r"([1-4])", 1).cast("double"),
        ).otherwise(F.lit(None).cast("double")),
    )
    .withColumn(
        "wifi_clean",
        F.when(wifi_text.contains("free"), F.lit("free"))
        .when(wifi_text.contains("paid"), F.lit("paid"))
        .when(wifi_text.isin("no", "none", "false", "0"), F.lit("no"))
        .otherwise(F.lit("unknown")),
    )
    .withColumn(
        "outdoor_seating_clean",
        F.when(outdoor_text.isin("true", "1", "yes"), F.lit("true"))
        .when(outdoor_text.isin("false", "0", "no"), F.lit("false"))
        .otherwise(F.lit("unknown")),
    )
    .withColumn(
        "is_open_clean",
        F.when(open_text.isin("1", "true", "yes", "open"), F.lit(1))
        .otherwise(F.lit(0)),
    )
    .withColumn("city_clean", F.lower(F.trim(F.col("city"))))
    .withColumn("state_clean", F.upper(F.trim(F.col("state"))))
)

business_df.select(
    "business_id",
    "price_range_numeric",
    "wifi_clean",
    "outdoor_seating_clean",
    "is_open_clean",
).show(20, truncate=False)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----------------------+-------------------+----------+---------------------+-------------+
|business_id           |price_range_numeric|wifi_clean|outdoor_seating_clean|is_open_clean|
+----------------------+-------------------+----------+---------------------+-------------+
|--ARBQr1WMsTWiwOKOj-FQ|2.0                |free      |true                 |0            |
|-0m4IwD1FIOqkA8dh4mVfQ|1.0                |unknown   |true                 |1            |
|-4E0hSCldRJZLI-1cT38Sw|2.0                |no        |false                |1            |
|-4bCujgnMeYCu5RSmy0xPQ|NULL               |unknown   |unknown              |1            |
|-4qjiD8Rk5yNDHM6H06A2Q|NULL               |unknown   |unknown              |1            |
|-53z4kzdbB9F4kPQqmQjyg|2.0                |unknown   |unknown              |0            |
|-5Rah4ZvWsDu4oilUZxhtw|2.0                |free      |false                |1            |
|-7KnD-G4ZYi7-Xs4ZJAYWQ|2.0                |free      |false                |1  

In [19]:
# ============================================================
# CELL 16: CREATE INFERENCE-READY BUSINESS DATAFRAME
# ============================================================

BUSINESS_SERVING_COLUMNS = [
    "business_id",
    "name",
    "categories",
    "category_tokens",
    "city",
    "city_clean",
    "state",
    "state_clean",
    "latitude",
    "longitude",
    "stars",
    "review_count",
    "is_open",
    "is_open_clean",
    "price_range_numeric",
    "wifi_clean",
    "outdoor_seating_clean",
    "category_vector_normalized",
]

business_serving_df = business_df.select(*BUSINESS_SERVING_COLUMNS).cache()
serving_row_count = business_serving_df.count()

if serving_row_count != business_row_count:
    raise ValueError(
        "Serving row count does not match the validated input row count."
    )

print("Inference-ready business rows:", serving_row_count)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Inference-ready business rows: 150346

In [20]:
# ============================================================
# CELL 17: QUERY CLEANING HELPERS
# ============================================================

def clean_text(value, uppercase=False):
    if value is None:
        return None
    cleaned_value = str(value).strip()
    if cleaned_value == "":
        return None
    return cleaned_value.upper() if uppercase else cleaned_value.lower()


def clean_boolean_attribute(value):
    if value is None:
        return None
    if isinstance(value, bool):
        return "true" if value else "false"

    cleaned_value = str(value).strip().lower()
    true_values = {"true", "yes", "1", "available"}
    false_values = {"false", "no", "0", "not available"}

    if cleaned_value in true_values:
        return "true"
    if cleaned_value in false_values:
        return "false"
    return cleaned_value


def clean_user_query(user_query):
    if not isinstance(user_query, dict):
        raise TypeError("USER_QUERY must be a dictionary.")

    cleaned_categories = []
    for category in user_query.get("categories", []):
        cleaned_category = clean_text(category)
        if cleaned_category is not None:
            cleaned_categories.append(cleaned_category)
    cleaned_categories = list(dict.fromkeys(cleaned_categories))

    location = user_query.get("location", {}) or {}
    cleaned_location = {
        "city": clean_text(location.get("city")),
        "state": clean_text(location.get("state"), uppercase=True),
    }

    cleaned_attributes = {}
    for attribute_name, attribute_value in (user_query.get("attributes", {}) or {}).items():
        cleaned_name = clean_text(attribute_name)
        if cleaned_name is None:
            continue

        if cleaned_name == "price_range":
            if attribute_value is not None:
                price_value = float(attribute_value)
                if price_value not in {1.0, 2.0, 3.0, 4.0}:
                    raise ValueError("price_range must be 1, 2, 3, or 4.")
                cleaned_attributes[cleaned_name] = price_value
        elif cleaned_name == "outdoor_seating":
            cleaned_value = clean_boolean_attribute(attribute_value)
            if cleaned_value not in {None, "true", "false"}:
                raise ValueError("outdoor_seating must be true/false or yes/no.")
            if cleaned_value is not None:
                cleaned_attributes[cleaned_name] = cleaned_value
        elif cleaned_name == "wifi":
            cleaned_value = clean_text(attribute_value)
            if cleaned_value not in {None, "free", "paid", "no"}:
                raise ValueError("wifi must be free, paid, or no.")
            if cleaned_value is not None:
                cleaned_attributes[cleaned_name] = cleaned_value
        else:
            raise ValueError(
                f"Unsupported attribute '{attribute_name}'. "
                "Supported attributes: price_range, wifi, outdoor_seating."
            )

    minimum_rating = user_query.get("minimum_rating")
    if minimum_rating is not None:
        minimum_rating = float(minimum_rating)
        if minimum_rating < 1.0 or minimum_rating > 5.0:
            raise ValueError("minimum_rating must be between 1.0 and 5.0.")

    top_k = int(user_query.get("top_k", 10))
    if top_k <= 0 or top_k > 100:
        raise ValueError("top_k must be between 1 and 100.")

    exclude_business_ids = [
        str(value).strip()
        for value in user_query.get("exclude_business_ids", [])
        if value is not None and str(value).strip()
    ]

    return {
        "categories": cleaned_categories,
        "location": cleaned_location,
        "attributes": cleaned_attributes,
        "minimum_rating": minimum_rating,
        "top_k": top_k,
        "exclude_business_ids": list(dict.fromkeys(exclude_business_ids)),
    }

print("Query cleaning helpers ready.")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Query cleaning helpers ready.

In [21]:
# ============================================================
# CELL 18: REUSABLE RECOMMENDATION FUNCTION
# ============================================================

@F.udf(returnType=DoubleType())
def vector_dot_product(left_vector, right_vector):
    if left_vector is None or right_vector is None:
        return 0.0
    return float(left_vector.dot(right_vector))


def recommend_businesses(
    user_query,
    business_features_df,
    vectorizer_model,
    normalizer_model,
    weights,
):
    """Return a lazy Spark DataFrame containing ranked recommendations."""

    cleaned_query = clean_user_query(user_query)

    query_schema = StructType([
        StructField("category_tokens", ArrayType(StringType()), nullable=False)
    ])
    query_category_df = spark.createDataFrame(
        [(cleaned_query["categories"],)], schema=query_schema
    )

    query_category_df = vectorizer_model.transform(query_category_df)
    query_category_df = normalizer_model.transform(query_category_df)
    query_vector = query_category_df.select(
        "category_vector_normalized"
    ).first()["category_vector_normalized"]

    query_vector_df = spark.createDataFrame(
        [(query_vector,)],
        StructType([
            StructField("query_category_vector", VectorUDT(), nullable=False)
        ]),
    )

    candidates_df = business_features_df.filter(F.col("is_open_clean") == 1)

    city_value = cleaned_query["location"]["city"]
    state_value = cleaned_query["location"]["state"]
    minimum_rating = cleaned_query["minimum_rating"]

    if city_value is not None:
        candidates_df = candidates_df.filter(F.col("city_clean") == city_value)
    if state_value is not None:
        candidates_df = candidates_df.filter(F.col("state_clean") == state_value)
    if minimum_rating is not None:
        candidates_df = candidates_df.filter(F.col("stars") >= minimum_rating)

    excluded_ids = cleaned_query["exclude_business_ids"]
    if excluded_ids:
        candidates_df = candidates_df.filter(~F.col("business_id").isin(excluded_ids))

    requested_attributes = cleaned_query["attributes"]
    attribute_match_expressions = []

    if "price_range" in requested_attributes:
        attribute_match_expressions.append(
            F.when(
                F.col("price_range_numeric") == F.lit(requested_attributes["price_range"]),
                F.lit(1),
            ).otherwise(F.lit(0))
        )
    if "wifi" in requested_attributes:
        attribute_match_expressions.append(
            F.when(
                F.col("wifi_clean") == F.lit(requested_attributes["wifi"]),
                F.lit(1),
            ).otherwise(F.lit(0))
        )
    if "outdoor_seating" in requested_attributes:
        attribute_match_expressions.append(
            F.when(
                F.col("outdoor_seating_clean")
                == F.lit(requested_attributes["outdoor_seating"]),
                F.lit(1),
            ).otherwise(F.lit(0))
        )

    requested_attribute_count = len(attribute_match_expressions)
    matched_attribute_count_expression = F.lit(0)
    for expression in attribute_match_expressions:
        matched_attribute_count_expression = (
            matched_attribute_count_expression + expression
        )

    max_review_log = (
        business_features_df.select(
            F.max(F.log1p(F.col("review_count"))).alias("max_review_log")
        ).first()["max_review_log"]
        or 1.0
    )

    scored_df = (
        candidates_df
        .crossJoin(F.broadcast(query_vector_df))
        .withColumn(
            "category_match_score",
            vector_dot_product(
                F.col("category_vector_normalized"),
                F.col("query_category_vector"),
            ),
        )
        .withColumn(
            "requested_attribute_count",
            F.lit(requested_attribute_count),
        )
        .withColumn(
            "matched_attribute_count",
            matched_attribute_count_expression,
        )
        .withColumn(
            "attribute_match_score",
            F.when(
                F.col("requested_attribute_count") > 0,
                F.col("matched_attribute_count")
                / F.col("requested_attribute_count"),
            ).otherwise(F.lit(0.0)),
        )
        .withColumn(
            "all_requested_attributes_matched",
            F.when(
                (F.col("requested_attribute_count") == 0)
                | (
                    F.col("matched_attribute_count")
                    == F.col("requested_attribute_count")
                ),
                F.lit(1),
            ).otherwise(F.lit(0)),
        )
        .withColumn("rating_score", F.col("stars") / F.lit(5.0))
        .withColumn(
            "review_confidence_score",
            F.log1p(F.col("review_count")) / F.lit(float(max_review_log)),
        )
        .withColumn(
            "final_score",
            F.col("category_match_score") * F.lit(float(weights["category"]))
            + F.col("attribute_match_score") * F.lit(float(weights["attributes"]))
            + F.col("rating_score") * F.lit(float(weights["rating"]))
            + F.col("review_confidence_score")
            * F.lit(float(weights["review_confidence"])),
        )
        .withColumn(
            "attribute_match_type",
            F.when(
                F.col("requested_attribute_count") == 0,
                F.lit("No attributes requested"),
            )
            .when(
                F.col("all_requested_attributes_matched") == 1,
                F.lit("Exact attribute match"),
            )
            .when(
                F.col("matched_attribute_count") > 0,
                F.lit("Partial attribute match"),
            )
            .otherwise(F.lit("No attribute match")),
        )
    )

    ranking_window = Window.orderBy(
        F.col("all_requested_attributes_matched").desc(),
        F.col("final_score").desc(),
        F.col("stars").desc(),
        F.col("review_count").desc(),
        F.col("business_id").asc(),
    )

    top_k = cleaned_query["top_k"]

    return (
        scored_df
        .withColumn("recommendation_rank", F.row_number().over(ranking_window))
        .filter(F.col("recommendation_rank") <= F.lit(top_k))
        .withColumn(
            "category_match_percentage",
            F.round(F.col("category_match_score") * 100, 2),
        )
        .withColumn(
            "attribute_match_percentage",
            F.round(F.col("attribute_match_score") * 100, 2),
        )
        .withColumn(
            "final_score_percentage",
            F.round(F.col("final_score") * 100, 2),
        )
        .orderBy("recommendation_rank")
    )

print("Reusable recommendation function ready.")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Reusable recommendation function ready.

In [22]:
# ============================================================
# CELL 19: FLEXIBLE USER SEARCH QUERY
# ============================================================

USER_QUERY = {
    "categories": ["pizza", "italian"],
    "location": {
        "city": None,
        "state": None,
    },
    "attributes": {
        "price_range": 2.0,
        "wifi": "free",
        "outdoor_seating": True,
    },
    "minimum_rating": 4.0,
    "top_k": 10,
    "exclude_business_ids": [],
}

CLEANED_USER_QUERY = clean_user_query(USER_QUERY)
print(json.dumps(CLEANED_USER_QUERY, indent=2))


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

{
  "categories": [
    "pizza",
    "italian"
  ],
  "location": {
    "city": null,
    "state": null
  },
  "attributes": {
    "price_range": 2.0,
    "wifi": "free",
    "outdoor_seating": "true"
  },
  "minimum_rating": 4.0,
  "top_k": 10,
  "exclude_business_ids": []
}

In [23]:
# ============================================================
# CELL 20: INITIAL MODEL WEIGHTS AND QUERY TEST
# ============================================================
# These weights are only the starting configuration. The
# validation grid below selects the final stored weights.

INITIAL_WEIGHTS = {
    "category": 0.50,
    "attributes": 0.25,
    "rating": 0.15,
    "review_confidence": 0.10,
}

initial_recommendations_df = recommend_businesses(
    user_query=USER_QUERY,
    business_features_df=business_serving_df,
    vectorizer_model=category_vectorizer_model,
    normalizer_model=normalizer,
    weights=INITIAL_WEIGHTS,
)

initial_recommendations_df.select(
    "recommendation_rank",
    "business_id",
    "name",
    "categories",
    "city",
    "state",
    "stars",
    "review_count",
    "price_range_numeric",
    "wifi_clean",
    "outdoor_seating_clean",
    "category_match_percentage",
    "attribute_match_percentage",
    "final_score_percentage",
).show(USER_QUERY["top_k"], truncate=False)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-------------------+----------------------+----+---------------------------+-------------+-----+-----+------------+-------------------+----------+---------------------+-------------------------+--------------------------+----------------------+
|recommendation_rank|business_id           |name|categories                 |city         |state|stars|review_count|price_range_numeric|wifi_clean|outdoor_seating_clean|category_match_percentage|attribute_match_percentage|final_score_percentage|
+-------------------+----------------------+----+---------------------------+-------------+-----+-----+------------+-------------------+----------+---------------------+-------------------------+--------------------------+----------------------+
|1                  |VOcGcN0bvGU_nzxbJgR5jQ|NULL|Restaurants, Italian, Pizza|Santa Barbara|CA   |4.5  |311         |2.0                |free      |true                 |81.65                    |100.0                     |85.75                 |
|2              

## Offline evaluation and weight tuning

The item representation is fitted on the complete business catalogue because it is unsupervised metadata encoding. To avoid selecting and reporting scoring weights on the same examples, generated search queries are split into **70% train/reference**, **15% validation**, and **15% test** partitions. Validation selects the weight combination; test is used once for the final metrics.


In [24]:
# ============================================================
# CELL 21: OFFLINE EVALUATION CONFIGURATION
# ============================================================

RANDOM_SEED = 42
EVALUATION_MINIMUM_RATING = 4.0
MINIMUM_RELEVANT_BUSINESSES = 5
MAXIMUM_RELEVANT_BUSINESSES = 500
MAX_QUERIES_PER_TYPE = 20
EVALUATION_TOP_K = 10

TRAIN_RATIO = 0.70
VALIDATION_RATIO = 0.15
TEST_RATIO = 0.15

print("Offline evaluation configuration ready.")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Offline evaluation configuration ready.

In [25]:
# ============================================================
# CELL 22: CREATE OFFLINE EVALUATION QUERIES
# ============================================================

eligible_business_df = (
    business_serving_df
    .filter(F.col("is_open_clean") == 1)
    .filter(F.col("stars") >= F.lit(EVALUATION_MINIMUM_RATING))
    .filter(F.size(F.col("category_tokens")) > 0)
)

exploded_category_df = (
    eligible_business_df
    .select(
        "business_id",
        "state_clean",
        "price_range_numeric",
        "wifi_clean",
        "outdoor_seating_clean",
        F.explode("category_tokens").alias("query_category"),
    )
    .filter(~F.col("query_category").isin("unknown", "restaurants"))
)


def limit_query_type(dataframe, query_type, order_columns):
    return (
        dataframe
        .filter(
            (F.col("relevant_business_count") >= MINIMUM_RELEVANT_BUSINESSES)
            & (F.col("relevant_business_count") <= MAXIMUM_RELEVANT_BUSINESSES)
        )
        .orderBy(*order_columns)
        .limit(MAX_QUERIES_PER_TYPE)
        .withColumn("query_type", F.lit(query_type))
    )

category_state_queries_df = limit_query_type(
    exploded_category_df
    .groupBy("query_category", "state_clean")
    .agg(F.countDistinct("business_id").alias("relevant_business_count"))
    .withColumn("query_price_range", F.lit(None).cast("double"))
    .withColumn("query_wifi", F.lit(None).cast("string"))
    .withColumn("query_outdoor_seating", F.lit(None).cast("string")),
    "category_state",
    [F.desc("relevant_business_count"), F.asc("state_clean"), F.asc("query_category")],
)

category_price_queries_df = limit_query_type(
    exploded_category_df
    .filter(F.col("price_range_numeric").isNotNull())
    .groupBy("query_category", "state_clean", "price_range_numeric")
    .agg(F.countDistinct("business_id").alias("relevant_business_count"))
    .withColumnRenamed("price_range_numeric", "query_price_range")
    .withColumn("query_wifi", F.lit(None).cast("string"))
    .withColumn("query_outdoor_seating", F.lit(None).cast("string")),
    "category_state_price",
    [F.desc("relevant_business_count"), F.asc("state_clean"), F.asc("query_category")],
)

category_wifi_queries_df = limit_query_type(
    exploded_category_df
    .filter(~F.col("wifi_clean").isin("unknown"))
    .groupBy("query_category", "state_clean", "wifi_clean")
    .agg(F.countDistinct("business_id").alias("relevant_business_count"))
    .withColumnRenamed("wifi_clean", "query_wifi")
    .withColumn("query_price_range", F.lit(None).cast("double"))
    .withColumn("query_outdoor_seating", F.lit(None).cast("string")),
    "category_state_wifi",
    [F.desc("relevant_business_count"), F.asc("state_clean"), F.asc("query_category")],
)

category_outdoor_queries_df = limit_query_type(
    exploded_category_df
    .filter(~F.col("outdoor_seating_clean").isin("unknown"))
    .groupBy("query_category", "state_clean", "outdoor_seating_clean")
    .agg(F.countDistinct("business_id").alias("relevant_business_count"))
    .withColumnRenamed("outdoor_seating_clean", "query_outdoor_seating")
    .withColumn("query_price_range", F.lit(None).cast("double"))
    .withColumn("query_wifi", F.lit(None).cast("string")),
    "category_state_outdoor",
    [F.desc("relevant_business_count"), F.asc("state_clean"), F.asc("query_category")],
)

query_columns = [
    "query_type",
    "query_category",
    "state_clean",
    "query_price_range",
    "query_wifi",
    "query_outdoor_seating",
    "relevant_business_count",
]

evaluation_queries_df = (
    category_state_queries_df.select(*query_columns)
    .unionByName(category_price_queries_df.select(*query_columns))
    .unionByName(category_wifi_queries_df.select(*query_columns))
    .unionByName(category_outdoor_queries_df.select(*query_columns))
    .withColumn(
        "query_id",
        F.sha2(
            F.concat_ws(
                "|",
                F.col("query_type"),
                F.col("query_category"),
                F.col("state_clean"),
                F.coalesce(F.col("query_price_range").cast("string"), F.lit("")),
                F.coalesce(F.col("query_wifi"), F.lit("")),
                F.coalesce(F.col("query_outdoor_seating"), F.lit("")),
            ),
            256,
        ),
    )
    .dropDuplicates(["query_id"])
    .cache()
)

evaluation_query_count = evaluation_queries_df.count()
if evaluation_query_count < 10:
    raise ValueError(
        f"Only {evaluation_query_count} evaluation queries were created. "
        "Reduce MINIMUM_RELEVANT_BUSINESSES or inspect the input data."
    )

print("Evaluation query count:", evaluation_query_count)
evaluation_queries_df.groupBy("query_type").count().show(truncate=False)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Evaluation query count: 80
+----------------------+-----+
|query_type            |count|
+----------------------+-----+
|category_state_price  |20   |
|category_state_wifi   |20   |
|category_state        |20   |
|category_state_outdoor|20   |
+----------------------+-----+

In [26]:
# ============================================================
# CELL 23: DETERMINISTIC 70/15/15 QUERY SPLIT
# ============================================================

split_window = Window.orderBy(F.rand(RANDOM_SEED), F.col("query_id"))
ranked_queries_df = evaluation_queries_df.withColumn(
    "split_rank", F.row_number().over(split_window)
)

train_end = max(1, int(evaluation_query_count * TRAIN_RATIO))
validation_end = max(train_end + 1, int(evaluation_query_count * (TRAIN_RATIO + VALIDATION_RATIO)))
validation_end = min(validation_end, evaluation_query_count - 1)

split_queries_df = ranked_queries_df.withColumn(
    "dataset_split",
    F.when(F.col("split_rank") <= train_end, F.lit("train"))
    .when(F.col("split_rank") <= validation_end, F.lit("validation"))
    .otherwise(F.lit("test")),
).cache()

train_queries_df = split_queries_df.filter(F.col("dataset_split") == "train")
validation_queries_df = split_queries_df.filter(F.col("dataset_split") == "validation")
test_queries_df = split_queries_df.filter(F.col("dataset_split") == "test")

split_counts = {
    row["dataset_split"]: row["count"]
    for row in split_queries_df.groupBy("dataset_split").count().collect()
}

if split_counts.get("validation", 0) == 0 or split_counts.get("test", 0) == 0:
    raise ValueError("Validation or test query split is empty.")

print("Query split counts:", split_counts)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Query split counts: {'train': 56, 'validation': 12, 'test': 12}

In [27]:
# ============================================================
# CELL 24: BUILD QUERY VECTORS AND EVALUATION CANDIDATES
# ============================================================

query_vector_input_df = split_queries_df.withColumn(
    "category_tokens", F.array(F.col("query_category"))
)
query_vector_df = category_vectorizer_model.transform(query_vector_input_df)
query_vector_df = normalizer.transform(query_vector_df)
query_vector_df = query_vector_df.withColumnRenamed(
    "category_vector_normalized", "query_category_vector_normalized"
)

max_review_log = (
    business_serving_df.select(
        F.max(F.log1p(F.col("review_count"))).alias("max_review_log")
    ).first()["max_review_log"]
    or 1.0
)

query_side_df = query_vector_df.select(
    "query_id",
    "query_type",
    "dataset_split",
    "query_category",
    "state_clean",
    "query_price_range",
    "query_wifi",
    "query_outdoor_seating",
    "query_category_vector_normalized",
)

business_side_df = business_serving_df.select(
    "business_id",
    "category_tokens",
    F.col("state_clean").alias("business_state_clean"),
    "stars",
    "review_count",
    "is_open_clean",
    "price_range_numeric",
    "wifi_clean",
    "outdoor_seating_clean",
    "category_vector_normalized",
)

evaluation_candidates_df = (
    business_side_df
    .filter(F.col("is_open_clean") == 1)
    .join(
        F.broadcast(query_side_df),
        F.col("business_state_clean") == F.col("state_clean"),
        "inner",
    )
    .withColumn(
        "category_match_score",
        vector_dot_product(
            F.col("category_vector_normalized"),
            F.col("query_category_vector_normalized"),
        ),
    )
    .withColumn(
        "requested_attribute_count",
        F.col("query_price_range").isNotNull().cast("int")
        + F.col("query_wifi").isNotNull().cast("int")
        + F.col("query_outdoor_seating").isNotNull().cast("int"),
    )
    .withColumn(
        "matched_attribute_count",
        F.when(
            F.col("query_price_range").isNotNull()
            & (F.col("price_range_numeric") == F.col("query_price_range")),
            F.lit(1),
        ).otherwise(F.lit(0))
        + F.when(
            F.col("query_wifi").isNotNull()
            & (F.col("wifi_clean") == F.col("query_wifi")),
            F.lit(1),
        ).otherwise(F.lit(0))
        + F.when(
            F.col("query_outdoor_seating").isNotNull()
            & (
                F.col("outdoor_seating_clean")
                == F.col("query_outdoor_seating")
            ),
            F.lit(1),
        ).otherwise(F.lit(0)),
    )
    .withColumn(
        "attribute_match_score",
        F.when(
            F.col("requested_attribute_count") > 0,
            F.col("matched_attribute_count") / F.col("requested_attribute_count"),
        ).otherwise(F.lit(0.0)),
    )
    .withColumn("rating_score", F.col("stars") / F.lit(5.0))
    .withColumn(
        "review_confidence_score",
        F.log1p(F.col("review_count")) / F.lit(float(max_review_log)),
    )
    .withColumn(
        "is_relevant",
        (
            F.expr("array_contains(category_tokens, query_category)")
            & (F.col("stars") >= F.lit(EVALUATION_MINIMUM_RATING))
            & (
                F.col("query_price_range").isNull()
                | (F.col("price_range_numeric") == F.col("query_price_range"))
            )
            & (
                F.col("query_wifi").isNull()
                | (F.col("wifi_clean") == F.col("query_wifi"))
            )
            & (
                F.col("query_outdoor_seating").isNull()
                | (
                    F.col("outdoor_seating_clean")
                    == F.col("query_outdoor_seating")
                )
            )
        ).cast("int"),
    )
    .select(
        "query_id",
        "query_type",
        "dataset_split",
        "business_id",
        "stars",
        "review_count",
        "category_match_score",
        "attribute_match_score",
        "rating_score",
        "review_confidence_score",
        "is_relevant",
    )
    .cache()
)

evaluation_candidate_count = evaluation_candidates_df.count()
if evaluation_candidate_count == 0:
    raise ValueError("No evaluation candidates were created.")

print("Evaluation candidate rows:", evaluation_candidate_count)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Evaluation candidate rows: 1381730

In [28]:
# ============================================================
# CELL 25: EVALUATION METRIC HELPERS
# ============================================================

@F.udf(returnType=DoubleType())
def ideal_dcg_at_k(relevant_count):
    if relevant_count is None or relevant_count <= 0:
        return 0.0
    ideal_hits = min(int(relevant_count), int(EVALUATION_TOP_K))
    return float(
        sum(1.0 / math.log2(rank + 1.0) for rank in range(1, ideal_hits + 1))
    )


def score_and_evaluate(candidate_df, weights):
    scored_df = candidate_df.withColumn(
        "final_score",
        F.col("category_match_score") * F.lit(float(weights["category"]))
        + F.col("attribute_match_score") * F.lit(float(weights["attributes"]))
        + F.col("rating_score") * F.lit(float(weights["rating"]))
        + F.col("review_confidence_score")
        * F.lit(float(weights["review_confidence"])),
    )

    ranking_window = Window.partitionBy("query_id").orderBy(
        F.col("final_score").desc(),
        F.col("stars").desc(),
        F.col("review_count").desc(),
        F.col("business_id").asc(),
    )

    top_k_df = (
        scored_df
        .withColumn("recommendation_rank", F.row_number().over(ranking_window))
        .filter(F.col("recommendation_rank") <= EVALUATION_TOP_K)
        .cache()
    )

    relevant_counts_df = (
        candidate_df
        .filter(F.col("is_relevant") == 1)
        .groupBy("query_id", "query_type")
        .agg(F.countDistinct("business_id").alias("relevant_count"))
    )

    hit_summary_df = top_k_df.groupBy("query_id", "query_type").agg(
        F.sum("is_relevant").cast("double").alias("hit_count"),
        F.min(
            F.when(F.col("is_relevant") == 1, F.col("recommendation_rank"))
        ).alias("first_relevant_rank"),
        F.sum(
            F.when(
                F.col("is_relevant") == 1,
                F.lit(1.0)
                / (
                    F.log(F.col("recommendation_rank") + F.lit(1.0))
                    / F.log(F.lit(2.0))
                ),
            ).otherwise(F.lit(0.0))
        ).alias("dcg_at_k"),
    )

    query_metrics_df = (
        relevant_counts_df
        .join(hit_summary_df, ["query_id", "query_type"], "left")
        .fillna({"hit_count": 0.0, "dcg_at_k": 0.0})
        .withColumn(
            "precision_at_k",
            F.col("hit_count") / F.lit(float(EVALUATION_TOP_K)),
        )
        .withColumn(
            "recall_at_k",
            F.when(
                F.col("relevant_count") > 0,
                F.col("hit_count") / F.col("relevant_count"),
            ).otherwise(F.lit(0.0)),
        )
        .withColumn(
            "hit_rate_at_k",
            F.when(F.col("hit_count") > 0, F.lit(1.0)).otherwise(F.lit(0.0)),
        )
        .withColumn(
            "mrr_at_k",
            F.when(
                F.col("first_relevant_rank").isNotNull(),
                F.lit(1.0) / F.col("first_relevant_rank"),
            ).otherwise(F.lit(0.0)),
        )
        .withColumn("idcg_at_k", ideal_dcg_at_k(F.col("relevant_count")))
        .withColumn(
            "ndcg_at_k",
            F.when(
                F.col("idcg_at_k") > 0,
                F.col("dcg_at_k") / F.col("idcg_at_k"),
            ).otherwise(F.lit(0.0)),
        )
    )

    overall_metrics_df = query_metrics_df.agg(
        F.round(F.avg("precision_at_k"), 4).alias("precision_at_k"),
        F.round(F.avg("recall_at_k"), 4).alias("recall_at_k"),
        F.round(F.avg("hit_rate_at_k"), 4).alias("hit_rate_at_k"),
        F.round(F.avg("mrr_at_k"), 4).alias("mrr_at_k"),
        F.round(F.avg("ndcg_at_k"), 4).alias("ndcg_at_k"),
        F.countDistinct("query_id").alias("query_count"),
    )

    return top_k_df, query_metrics_df, overall_metrics_df

print("Evaluation helpers ready.")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Evaluation helpers ready.

In [29]:
# ============================================================
# CELL 26: COMPACT VALIDATION GRID SEARCH
# ============================================================

WEIGHT_GRID = [
    {"category": 0.55, "attributes": 0.20, "rating": 0.15, "review_confidence": 0.10},
    {"category": 0.50, "attributes": 0.25, "rating": 0.15, "review_confidence": 0.10},
    {"category": 0.60, "attributes": 0.15, "rating": 0.15, "review_confidence": 0.10},
    {"category": 0.50, "attributes": 0.20, "rating": 0.20, "review_confidence": 0.10},
    {"category": 0.45, "attributes": 0.30, "rating": 0.15, "review_confidence": 0.10},
    {"category": 0.50, "attributes": 0.25, "rating": 0.10, "review_confidence": 0.15},
]

for weight_set in WEIGHT_GRID:
    if abs(sum(weight_set.values()) - 1.0) > 1e-9:
        raise ValueError(f"Weights do not sum to 1.0: {weight_set}")

validation_candidate_df = evaluation_candidates_df.filter(
    F.col("dataset_split") == "validation"
).cache()

validation_rows = []

for grid_index, weight_set in enumerate(WEIGHT_GRID, start=1):
    validation_top_k_df, _, validation_metrics_df = score_and_evaluate(
        validation_candidate_df,
        weight_set,
    )
    metric_row = validation_metrics_df.first()
    validation_top_k_df.unpersist()

    if metric_row is None or metric_row["query_count"] == 0:
        raise ValueError("Validation evaluation returned no query metrics.")

    validation_rows.append({
        "grid_index": grid_index,
        "category_weight": float(weight_set["category"]),
        "attribute_weight": float(weight_set["attributes"]),
        "rating_weight": float(weight_set["rating"]),
        "review_weight": float(weight_set["review_confidence"]),
        "precision_at_k": float(metric_row["precision_at_k"]),
        "recall_at_k": float(metric_row["recall_at_k"]),
        "hit_rate_at_k": float(metric_row["hit_rate_at_k"]),
        "mrr_at_k": float(metric_row["mrr_at_k"]),
        "ndcg_at_k": float(metric_row["ndcg_at_k"]),
        "validation_queries": int(metric_row["query_count"]),
    })

weight_results_df = spark.createDataFrame(validation_rows)
weight_results_df.orderBy(
    F.desc("ndcg_at_k"),
    F.desc("mrr_at_k"),
    F.desc("recall_at_k"),
    F.asc("grid_index"),
).show(truncate=False)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----------------+---------------+----------+-------------+--------+---------+--------------+-------------+-----------+-------------+------------------+
|attribute_weight|category_weight|grid_index|hit_rate_at_k|mrr_at_k|ndcg_at_k|precision_at_k|rating_weight|recall_at_k|review_weight|validation_queries|
+----------------+---------------+----------+-------------+--------+---------+--------------+-------------+-----------+-------------+------------------+
|0.2             |0.5            |4         |1.0          |0.9583  |0.9519   |0.95          |0.2          |0.0263     |0.1          |12                |
|0.2             |0.55           |1         |1.0          |0.9583  |0.9401   |0.9417        |0.15         |0.0261     |0.1          |12                |
|0.25            |0.5            |2         |1.0          |0.9583  |0.9401   |0.9417        |0.15         |0.0261     |0.1          |12                |
|0.3             |0.45           |5         |1.0          |0.9583  |0.9401   |0.94

In [30]:
# ============================================================
# CELL 27: SELECT THE BEST VALIDATION WEIGHTS
# ============================================================

best_weight_row = (
    weight_results_df
    .orderBy(
        F.desc("ndcg_at_k"),
        F.desc("mrr_at_k"),
        F.desc("recall_at_k"),
        F.asc("grid_index"),
    )
    .first()
)

if best_weight_row is None:
    raise ValueError("No best weight combination was selected.")

BEST_CATEGORY_WEIGHT = float(best_weight_row["category_weight"])
BEST_ATTRIBUTE_WEIGHT = float(best_weight_row["attribute_weight"])
BEST_RATING_WEIGHT = float(best_weight_row["rating_weight"])
BEST_REVIEW_WEIGHT = float(best_weight_row["review_weight"])

BEST_WEIGHTS = {
    "category": BEST_CATEGORY_WEIGHT,
    "attributes": BEST_ATTRIBUTE_WEIGHT,
    "rating": BEST_RATING_WEIGHT,
    "review_confidence": BEST_REVIEW_WEIGHT,
}

print("Selected weights:")
print(json.dumps(BEST_WEIGHTS, indent=2))


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Selected weights:
{
  "category": 0.5,
  "attributes": 0.2,
  "rating": 0.2,
  "review_confidence": 0.1
}

In [31]:
# ============================================================
# CELL 28: FINAL TEST EVALUATION — RUN ONCE
# ============================================================

test_candidate_df = evaluation_candidates_df.filter(
    F.col("dataset_split") == "test"
).cache()

test_top_k_df, query_test_metrics_df, final_test_metrics_df = score_and_evaluate(
    test_candidate_df,
    BEST_WEIGHTS,
)

print("Final model evaluation results")
print("------------------------------")
final_test_metrics_df.show(truncate=False)

print("Test metrics by query type")
query_test_metrics_df.groupBy("query_type").agg(
    F.round(F.avg("precision_at_k"), 4).alias("precision_at_k"),
    F.round(F.avg("recall_at_k"), 4).alias("recall_at_k"),
    F.round(F.avg("mrr_at_k"), 4).alias("mrr_at_k"),
    F.round(F.avg("ndcg_at_k"), 4).alias("ndcg_at_k"),
).orderBy("query_type").show(truncate=False)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Final model evaluation results
------------------------------
+--------------+-----------+-------------+--------+---------+-----------+
|precision_at_k|recall_at_k|hit_rate_at_k|mrr_at_k|ndcg_at_k|query_count|
+--------------+-----------+-------------+--------+---------+-----------+
|0.9           |0.0286     |1.0          |0.9167  |0.905    |12         |
+--------------+-----------+-------------+--------+---------+-----------+

Test metrics by query type
+----------------------+--------------+-----------+--------+---------+
|query_type            |precision_at_k|recall_at_k|mrr_at_k|ndcg_at_k|
+----------------------+--------------+-----------+--------+---------+
|category_state        |1.0           |0.0218     |1.0     |1.0      |
|category_state_outdoor|0.8           |0.0303     |0.875   |0.8231   |
|category_state_price  |0.95          |0.0301     |0.75    |0.89     |
|category_state_wifi   |0.925         |0.0296     |1.0     |0.9469   |
+----------------------+--------------+----

In [32]:
# ============================================================
# CELL 29: RECOMMENDATION COVERAGE
# ============================================================

candidate_business_count = (
    test_candidate_df.select("business_id").distinct().count()
)
recommended_business_count = (
    test_top_k_df.select("business_id").distinct().count()
)

coverage = (
    recommended_business_count / candidate_business_count
    if candidate_business_count > 0
    else 0.0
)

print("Candidate businesses:", candidate_business_count)
print("Unique recommended businesses:", recommended_business_count)
print("Catalogue coverage:", round(coverage, 4))
print("Catalogue coverage percentage:", round(coverage * 100, 2))


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Candidate businesses: 79490
Unique recommended businesses: 119
Catalogue coverage: 0.0015
Catalogue coverage percentage: 0.15

In [33]:
# ============================================================
# CELL 30: FINAL TUNED USER RECOMMENDATIONS
# ============================================================

final_recommendations_df = recommend_businesses(
    user_query=USER_QUERY,
    business_features_df=business_serving_df,
    vectorizer_model=category_vectorizer_model,
    normalizer_model=normalizer,
    weights=BEST_WEIGHTS,
).cache()

final_recommendation_count = final_recommendations_df.count()
if final_recommendation_count == 0:
    raise ValueError(
        "The configured USER_QUERY returned no recommendation. "
        "Relax location, rating, or attribute constraints."
    )

final_recommendations_df.select(
    "recommendation_rank",
    "business_id",
    "name",
    "categories",
    "city",
    "state",
    "stars",
    "review_count",
    "price_range_numeric",
    "wifi_clean",
    "outdoor_seating_clean",
    "category_match_percentage",
    "matched_attribute_count",
    "requested_attribute_count",
    "attribute_match_type",
    F.round(F.col("rating_score") * 100, 2).alias("rating_score_percentage"),
    F.round(
        F.col("review_confidence_score") * 100, 2
    ).alias("review_confidence_percentage"),
    "final_score_percentage",
).show(USER_QUERY["top_k"], truncate=False)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-------------------+----------------------+----+---------------------------+-------------+-----+-----+------------+-------------------+----------+---------------------+-------------------------+-----------------------+-------------------------+---------------------+-----------------------+----------------------------+----------------------+
|recommendation_rank|business_id           |name|categories                 |city         |state|stars|review_count|price_range_numeric|wifi_clean|outdoor_seating_clean|category_match_percentage|matched_attribute_count|requested_attribute_count|attribute_match_type |rating_score_percentage|review_confidence_percentage|final_score_percentage|
+-------------------+----------------------+----+---------------------------+-------------+-----+-----+------------+-------------------+----------+---------------------+-------------------------+-----------------------+-------------------------+---------------------+-----------------------+---------------------

In [34]:
# ============================================================
# CELL 31: PREPARE FINAL MODEL CONFIGURATION
# ============================================================

final_metrics = final_test_metrics_df.first().asDict()

FINAL_MODEL_CONFIG = {
    "model_name": "Yelp_Content_Based_Recommender",
    "model_version": MODEL_VERSION,
    "model_type": "Query-Based Content Recommender",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "spark_version": spark.version,
    "top_k": int(USER_QUERY["top_k"]),
    "weights": BEST_WEIGHTS,
    "test_metrics": {
        "precision_at_k": float(final_metrics["precision_at_k"]),
        "recall_at_k": float(final_metrics["recall_at_k"]),
        "hit_rate_at_k": float(final_metrics["hit_rate_at_k"]),
        "mrr_at_k": float(final_metrics["mrr_at_k"]),
        "ndcg_at_k": float(final_metrics["ndcg_at_k"]),
        "test_queries": int(final_metrics["query_count"]),
    },
    "coverage": {
        "candidate_businesses": int(candidate_business_count),
        "unique_recommended_businesses": int(recommended_business_count),
        "catalogue_coverage": round(float(coverage), 4),
        "catalogue_coverage_percentage": round(float(coverage) * 100, 2),
    },
    "feature_columns": [
        "categories",
        "city",
        "state",
        "stars",
        "review_count",
        "price_range_numeric",
        "wifi_clean",
        "outdoor_seating_clean",
    ],
    "vector_column": "category_vector_normalized",
    "input_path": BUSINESS_FEATURES_PATH,
    "business_vector_path": BUSINESS_VECTOR_OUTPUT_PATH,
    "category_vectorizer_path": CATEGORY_VECTORIZER_MODEL_PATH,
    "normalizer_path": NORMALIZER_MODEL_PATH,
}

print(json.dumps(FINAL_MODEL_CONFIG, indent=2))


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

{
  "model_name": "Yelp_Content_Based_Recommender",
  "model_version": "v1",
  "model_type": "Query-Based Content Recommender",
  "created_at_utc": "2026-08-01T18:17:58.873685+00:00",
  "spark_version": "3.5.3-amzn-0",
  "top_k": 10,
  "weights": {
    "category": 0.5,
    "attributes": 0.2,
    "rating": 0.2,
    "review_confidence": 0.1
  },
  "test_metrics": {
    "precision_at_k": 0.9,
    "recall_at_k": 0.0286,
    "hit_rate_at_k": 1.0,
    "mrr_at_k": 0.9167,
    "ndcg_at_k": 0.905,
    "test_queries": 12
  },
  "coverage": {
    "candidate_businesses": 79490,
    "unique_recommended_businesses": 119,
    "catalogue_coverage": 0.0015,
    "catalogue_coverage_percentage": 0.15
  },
  "feature_columns": [
    "categories",
    "city",
    "state",
    "stars",
    "review_count",
    "price_range_numeric",
    "wifi_clean",
    "outdoor_seating_clean"
  ],
  "vector_column": "category_vector_normalized",
  "input_path": "s3://yelpdataset-project/gold_layer/ml/content_based_filterin

## Save trained model and inference-ready data to S3

The following cell is the deployment boundary. It saves native Spark ML artifacts and precomputed business vectors. Recommendation scripts should load these outputs instead of fitting the model again.


In [35]:
# ============================================================
# CELL 32: VALIDATE FINAL SERVING DATA BEFORE SAVING
# ============================================================

REQUIRED_SERVING_COLUMNS = {
    "business_id",
    "categories",
    "category_tokens",
    "city",
    "state",
    "stars",
    "review_count",
    "is_open_clean",
    "price_range_numeric",
    "wifi_clean",
    "outdoor_seating_clean",
    "category_vector_normalized",
}

missing_serving_columns = sorted(
    REQUIRED_SERVING_COLUMNS - set(business_serving_df.columns)
)
if missing_serving_columns:
    raise ValueError(
        "Final serving data is missing columns: "
        + ", ".join(missing_serving_columns)
    )

if serving_row_count == 0:
    raise ValueError("Final serving data is empty.")

if business_serving_df.filter(F.col("business_id").isNull()).limit(1).count() > 0:
    raise ValueError("Final serving data contains a null business_id.")

print("Final serving data validation passed.")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Final serving data validation passed.

In [40]:
# ============================================================
# DEFINE OUTPUT PARTITIONS FOR S3 SAVING
# ============================================================

spark_parallelism = spark.sparkContext.defaultParallelism

# Dynamically select a reasonable number of output files.
OUTPUT_PARTITIONS = max(
    4,
    min(32, spark_parallelism)
)

print("Spark default parallelism:", spark_parallelism)
print("Output partitions:", OUTPUT_PARTITIONS)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Spark default parallelism: 6
Output partitions: 6

In [43]:
# ============================================================
# CELL 33: SAVE AND VERIFY ALL MODEL ARTIFACTS TO S3
# ============================================================

import traceback

# ------------------------------------------------------------
# Automatically configure output partitions if not defined
# ------------------------------------------------------------


# ------------------------------------------------------------
# More reliable change inside Cell 33
# To prevent this error in future automated runs,
#     add the following code near the beginning of Cell 33, immediately after:
# ------------------------------------------------------------

if "OUTPUT_PARTITIONS" not in globals():

    spark_parallelism = (
        spark.sparkContext.defaultParallelism
    )

    OUTPUT_PARTITIONS = max(
        4,
        min(32, spark_parallelism)
    )

    print(
        "OUTPUT_PARTITIONS was not defined. "
        "Automatically configured it as:",
        OUTPUT_PARTITIONS,
    )
else:
    print(
        "Using configured OUTPUT_PARTITIONS:",
        OUTPUT_PARTITIONS,
    )
# ------------------------------------------------------------
# 1. Validate that all required objects exist
# ------------------------------------------------------------

REQUIRED_SAVE_OBJECTS = [
    "category_vectorizer_model",
    "normalizer",
    "business_serving_df",
    "weight_results_df",
    "final_test_metrics_df",
    "final_recommendations_df",
    "FINAL_MODEL_CONFIG",
    "OUTPUT_PARTITIONS",
    "CONTENT_OUTPUT_PATH",
    "CATEGORY_VECTORIZER_MODEL_PATH",
    "NORMALIZER_MODEL_PATH",
    "BUSINESS_VECTOR_OUTPUT_PATH",
    "VALIDATION_RESULTS_OUTPUT_PATH",
    "TEST_METRICS_OUTPUT_PATH",
    "RECOMMENDATIONS_OUTPUT_PATH",
    "MODEL_CONFIG_OUTPUT_PATH",
    "path_exists",
    "delete_path_if_exists",
    "write_json_document",
]

missing_save_objects = [
    object_name
    for object_name in REQUIRED_SAVE_OBJECTS
    if object_name not in globals()
]

if missing_save_objects:
    raise NameError(
        "Cannot save the model because these objects are missing: "
        + ", ".join(missing_save_objects)
    )


# ------------------------------------------------------------
# 2. Validate all S3 output paths
# ------------------------------------------------------------

OUTPUT_PATHS = {
    "Category vectorizer": CATEGORY_VECTORIZER_MODEL_PATH,
    "Normalizer": NORMALIZER_MODEL_PATH,
    "Business vectors": BUSINESS_VECTOR_OUTPUT_PATH,
    "Validation results": VALIDATION_RESULTS_OUTPUT_PATH,
    "Test metrics": TEST_METRICS_OUTPUT_PATH,
    "Sample recommendations": RECOMMENDATIONS_OUTPUT_PATH,
    "Model configuration": MODEL_CONFIG_OUTPUT_PATH,
}

invalid_output_paths = {
    name: path
    for name, path in OUTPUT_PATHS.items()
    if not path.startswith("s3://")
}

if invalid_output_paths:
    raise ValueError(
        "The following output paths are not valid S3 paths: "
        + str(invalid_output_paths)
    )

print("Output root:", CONTENT_OUTPUT_PATH)


# ------------------------------------------------------------
# 3. Validate recommendation columns
# ------------------------------------------------------------

recommendation_columns_to_save = [
    "recommendation_rank",
    "business_id",
    "name",
    "categories",
    "city",
    "state",
    "stars",
    "review_count",
    "price_range_numeric",
    "wifi_clean",
    "outdoor_seating_clean",
    "category_match_score",
    "attribute_match_score",
    "rating_score",
    "review_confidence_score",
    "final_score",
    "final_score_percentage",
]

missing_recommendation_columns = [
    column_name
    for column_name in recommendation_columns_to_save
    if column_name not in final_recommendations_df.columns
]

if missing_recommendation_columns:
    raise ValueError(
        "Cannot save recommendations. Missing columns: "
        + ", ".join(missing_recommendation_columns)
    )


# ------------------------------------------------------------
# 4. Helper: clean an existing or partially written path
# ------------------------------------------------------------

def prepare_output_path(output_path):

    if path_exists(output_path):

        print("Existing or partial output found.")
        print("Deleting:", output_path)

        delete_path_if_exists(output_path)

        if path_exists(output_path):
            raise PermissionError(
                "Spark could not delete the existing output path. "
                "The EMR role probably does not have "
                "s3:DeleteObject permission for: "
                + output_path
            )


# ------------------------------------------------------------
# 5. Helper: execute and verify one saving operation
# ------------------------------------------------------------

def execute_save_step(step_number, step_name, output_path, save_function):

    print()
    print("=" * 70)
    print(f"STEP {step_number}: {step_name}")
    print("Output:", output_path)
    print("=" * 70)

    try:

        prepare_output_path(output_path)

        save_function()

        if not path_exists(output_path):
            raise RuntimeError(
                "The save command finished, but the output "
                "path could not be found."
            )

        print(f"SUCCESS: {step_name} saved.")

    except Exception as error:

        print()
        print("FAILED STEP:", step_name)
        print("ERROR TYPE:", type(error).__name__)
        print("ERROR MESSAGE:", str(error))
        print()
        print("Complete traceback:")
        traceback.print_exc()

        raise RuntimeError(
            f"Saving failed during: {step_name}"
        ) from error


# ------------------------------------------------------------
# 6. Test S3 write and delete permissions
# ------------------------------------------------------------

permission_test_path = (
    CONTENT_OUTPUT_PATH
    + "_s3_write_delete_permission_test/"
)

print()
print("=" * 70)
print("S3 WRITE AND DELETE PERMISSION TEST")
print("=" * 70)
print("Test path:", permission_test_path)

try:

    prepare_output_path(permission_test_path)

    (
        spark.range(1)
        .coalesce(1)
        .write
        .mode("errorifexists")
        .parquet(permission_test_path)
    )

    if not path_exists(permission_test_path):
        raise RuntimeError(
            "Spark completed the test write, "
            "but the test output was not found."
        )

    print("S3 write permission: PASSED")

    delete_path_if_exists(permission_test_path)

    if path_exists(permission_test_path):
        raise PermissionError(
            "S3 write succeeded, but deletion failed. "
            "The EMR role may be missing s3:DeleteObject."
        )

    print("S3 delete permission: PASSED")

except Exception as error:

    print()
    print("S3 permission test failed.")
    print("ERROR TYPE:", type(error).__name__)
    print("ERROR MESSAGE:", str(error))
    print()
    traceback.print_exc()

    raise RuntimeError(
        "EMR cannot safely write and overwrite model outputs "
        "in the configured S3 location."
    ) from error


# ------------------------------------------------------------
# 7. Save CountVectorizerModel
# ------------------------------------------------------------

execute_save_step(
    step_number=1,
    step_name="Category CountVectorizer model",
    output_path=CATEGORY_VECTORIZER_MODEL_PATH,
    save_function=lambda: (
        category_vectorizer_model
        .write()
        .save(CATEGORY_VECTORIZER_MODEL_PATH)
    ),
)


# ------------------------------------------------------------
# 8. Save Normalizer
# ------------------------------------------------------------

execute_save_step(
    step_number=2,
    step_name="L2 Normalizer model",
    output_path=NORMALIZER_MODEL_PATH,
    save_function=lambda: (
        normalizer
        .write()
        .save(NORMALIZER_MODEL_PATH)
    ),
)


# ------------------------------------------------------------
# 9. Save inference-ready business vectors
# ------------------------------------------------------------

execute_save_step(
    step_number=3,
    step_name="Inference-ready business vectors",
    output_path=BUSINESS_VECTOR_OUTPUT_PATH,
    save_function=lambda: (
        business_serving_df
        .repartition(OUTPUT_PARTITIONS)
        .write
        .mode("errorifexists")
        .option("compression", "snappy")
        .parquet(BUSINESS_VECTOR_OUTPUT_PATH)
    ),
)


# ------------------------------------------------------------
# 10. Save validation-grid results
# ------------------------------------------------------------

execute_save_step(
    step_number=4,
    step_name="Validation grid results",
    output_path=VALIDATION_RESULTS_OUTPUT_PATH,
    save_function=lambda: (
        weight_results_df
        .write
        .mode("errorifexists")
        .option("compression", "snappy")
        .parquet(VALIDATION_RESULTS_OUTPUT_PATH)
    ),
)


# ------------------------------------------------------------
# 11. Save final test metrics
# ------------------------------------------------------------

execute_save_step(
    step_number=5,
    step_name="Final test metrics",
    output_path=TEST_METRICS_OUTPUT_PATH,
    save_function=lambda: (
        final_test_metrics_df
        .write
        .mode("errorifexists")
        .option("compression", "snappy")
        .parquet(TEST_METRICS_OUTPUT_PATH)
    ),
)


# ------------------------------------------------------------
# 12. Save sample recommendations
# ------------------------------------------------------------

execute_save_step(
    step_number=6,
    step_name="Sample recommendations",
    output_path=RECOMMENDATIONS_OUTPUT_PATH,
    save_function=lambda: (
        final_recommendations_df
        .select(*recommendation_columns_to_save)
        .write
        .mode("errorifexists")
        .option("compression", "snappy")
        .parquet(RECOMMENDATIONS_OUTPUT_PATH)
    ),
)


# ------------------------------------------------------------
# 13. Save model configuration
# ------------------------------------------------------------

execute_save_step(
    step_number=7,
    step_name="Model configuration",
    output_path=MODEL_CONFIG_OUTPUT_PATH,
    save_function=lambda: write_json_document(
        FINAL_MODEL_CONFIG,
        MODEL_CONFIG_OUTPUT_PATH,
    ),
)


# ------------------------------------------------------------
# 14. Final verification summary
# ------------------------------------------------------------

print()
print("=" * 70)
print("ALL MODEL ARTIFACTS SAVED SUCCESSFULLY")
print("=" * 70)

for artifact_name, artifact_path in OUTPUT_PATHS.items():

    artifact_status = (
        "FOUND"
        if path_exists(artifact_path)
        else "MISSING"
    )

    print(
        f"{artifact_name}: "
        f"{artifact_status} -> {artifact_path}"
    )

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Using configured OUTPUT_PARTITIONS: 6
Output root: s3://yelpdataset-project/gold_layer/ml/content_based_recommender_model/v1/

S3 WRITE AND DELETE PERMISSION TEST
Test path: s3://yelpdataset-project/gold_layer/ml/content_based_recommender_model/v1/_s3_write_delete_permission_test/
S3 write permission: PASSED
S3 delete permission: PASSED

STEP 1: Category CountVectorizer model
Output: s3://yelpdataset-project/gold_layer/ml/content_based_recommender_model/v1/model_artifacts/category_vectorizer_model/
Existing or partial output found.
Deleting: s3://yelpdataset-project/gold_layer/ml/content_based_recommender_model/v1/model_artifacts/category_vectorizer_model/
SUCCESS: Category CountVectorizer model saved.

STEP 2: L2 Normalizer model
Output: s3://yelpdataset-project/gold_layer/ml/content_based_recommender_model/v1/model_artifacts/normalizer/
Existing or partial output found.
Deleting: s3://yelpdataset-project/gold_layer/ml/content_based_recommender_model/v1/model_artifacts/normalizer/
SUC

In [44]:
# ============================================================
# CELL 34: RELOAD SAVED ARTIFACTS FROM S3
# ============================================================

try:
    loaded_category_vectorizer_model = CountVectorizerModel.load(
        CATEGORY_VECTORIZER_MODEL_PATH
    )
    loaded_normalizer = Normalizer.load(NORMALIZER_MODEL_PATH)
    loaded_business_serving_df = spark.read.parquet(BUSINESS_VECTOR_OUTPUT_PATH)
    loaded_validation_results_df = spark.read.parquet(
        VALIDATION_RESULTS_OUTPUT_PATH
    )
    loaded_test_metrics_df = spark.read.parquet(TEST_METRICS_OUTPUT_PATH)
    loaded_recommendations_df = spark.read.parquet(
        RECOMMENDATIONS_OUTPUT_PATH
    )
    loaded_model_config = read_json_document(MODEL_CONFIG_OUTPUT_PATH)
except Exception as error:
    raise RuntimeError(
        "One or more saved model artifacts could not be reloaded from S3."
    ) from error

print("All saved artifacts reloaded successfully.")
print("Loaded vocabulary size:", len(loaded_category_vectorizer_model.vocabulary))


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

All saved artifacts reloaded successfully.
Loaded vocabulary size: 1256

In [45]:
# ============================================================
# CELL 35: SAVED-ARTIFACT CONSISTENCY CHECKS
# ============================================================

loaded_business_count = loaded_business_serving_df.count()

if loaded_business_count != serving_row_count:
    raise ValueError(
        f"Saved business row count {loaded_business_count} does not match "
        f"expected row count {serving_row_count}."
    )

if (
    loaded_category_vectorizer_model.vocabulary
    != category_vectorizer_model.vocabulary
):
    raise ValueError("Reloaded CountVectorizer vocabulary does not match.")

loaded_metric_row = loaded_test_metrics_df.first()
if loaded_metric_row is None:
    raise ValueError("Reloaded test metrics are empty.")

loaded_recommendation_count = loaded_recommendations_df.count()
if loaded_recommendation_count != final_recommendation_count:
    raise ValueError(
        "Reloaded recommendation row count does not match the saved output."
    )

if loaded_model_config.get("model_version") != MODEL_VERSION:
    raise ValueError("Reloaded model configuration has the wrong version.")

print("Saved-artifact consistency checks passed.")
print("Saved business rows:", loaded_business_count)
print("Saved recommendation rows:", loaded_recommendation_count)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Saved-artifact consistency checks passed.
Saved business rows: 150346
Saved recommendation rows: 10

In [46]:
# ============================================================
# CELL 36: SAVED-MODEL SMOKE TEST
# ============================================================
# This test deliberately uses only reloaded artifacts. It proves
# that a future script can generate recommendations without
# retraining the CountVectorizer.

SMOKE_TEST_QUERY = {
    "categories": ["pizza", "italian"],
    "location": {"city": None, "state": None},
    "attributes": {"price_range": 2.0},
    "minimum_rating": 4.0,
    "top_k": 5,
    "exclude_business_ids": [],
}

smoke_test_recommendations_df = recommend_businesses(
    user_query=SMOKE_TEST_QUERY,
    business_features_df=loaded_business_serving_df,
    vectorizer_model=loaded_category_vectorizer_model,
    normalizer_model=loaded_normalizer,
    weights=loaded_model_config["weights"],
).cache()

smoke_test_count = smoke_test_recommendations_df.count()
if smoke_test_count == 0:
    raise ValueError("Saved-model smoke test returned zero recommendations.")

smoke_test_recommendations_df.select(
    "recommendation_rank",
    "business_id",
    "name",
    "categories",
    "city",
    "state",
    "stars",
    "final_score_percentage",
).show(5, truncate=False)

print("Saved-model smoke test passed.")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-------------------+----------------------+----+---------------------------+-------------+-----+-----+----------------------+
|recommendation_rank|business_id           |name|categories                 |city         |state|stars|final_score_percentage|
+-------------------+----------------------+----+---------------------------+-------------+-----+-----+----------------------+
|1                  |hUl_zNzmwMd8TD8pPL81pg|NULL|Restaurants, Italian, Pizza|Metairie     |LA   |4.5  |85.48                 |
|2                  |VOcGcN0bvGU_nzxbJgR5jQ|NULL|Restaurants, Italian, Pizza|Santa Barbara|CA   |4.5  |85.25                 |
|3                  |YJVXMOIKhJjQczWj9YTwrg|NULL|Italian, Pizza, Restaurants|Clayton      |MO   |4.5  |84.92                 |
|4                  |ynAOBtqwYT9U4Jg79etHbg|NULL|Pizza, Italian, Restaurants|Glen Mills   |PA   |4.5  |84.87                 |
|5                  |EugP1H9WF04ClGhY83848A|NULL|Italian, Restaurants, Pizza|Riverview    |FL   |4.5  |84.85   

In [47]:
# ============================================================
# CELL 37: WRITE THE FINAL READY MANIFEST
# ============================================================
# Automation should consume this model version only when the
# manifest status is READY.

MODEL_MANIFEST = {
    "model_name": "Yelp_Content_Based_Recommender",
    "model_version": MODEL_VERSION,
    "status": "READY",
    "build_started_at_utc": BUILD_STARTED_AT_UTC,
    "build_completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "spark_version": spark.version,
    "spark_application_id": spark.sparkContext.applicationId,
    "input_path": BUSINESS_FEATURES_PATH,
    "output_root": CONTENT_OUTPUT_PATH,
    "input_business_rows": int(business_row_count),
    "saved_business_rows": int(loaded_business_count),
    "vocabulary_size": int(len(loaded_category_vectorizer_model.vocabulary)),
    "smoke_test_recommendations": int(smoke_test_count),
    "artifacts": {
        "business_vectors": BUSINESS_VECTOR_OUTPUT_PATH,
        "category_vectorizer": CATEGORY_VECTORIZER_MODEL_PATH,
        "normalizer": NORMALIZER_MODEL_PATH,
        "configuration": MODEL_CONFIG_OUTPUT_PATH,
        "validation_results": VALIDATION_RESULTS_OUTPUT_PATH,
        "test_metrics": TEST_METRICS_OUTPUT_PATH,
        "sample_recommendations": RECOMMENDATIONS_OUTPUT_PATH,
    },
}

write_json_document(MODEL_MANIFEST, MODEL_MANIFEST_OUTPUT_PATH)
verified_manifest = read_json_document(MODEL_MANIFEST_OUTPUT_PATH)

if verified_manifest.get("status") != "READY":
    raise ValueError("The final model manifest is not READY.")

print(json.dumps(verified_manifest, indent=2))
print("Final content-based recommendation workflow completed successfully.")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

{
  "artifacts": {
    "business_vectors": "s3://yelpdataset-project/gold_layer/ml/content_based_recommender_model/v1/business_feature_vectors/",
    "category_vectorizer": "s3://yelpdataset-project/gold_layer/ml/content_based_recommender_model/v1/model_artifacts/category_vectorizer_model/",
    "configuration": "s3://yelpdataset-project/gold_layer/ml/content_based_recommender_model/v1/model_configuration/",
    "normalizer": "s3://yelpdataset-project/gold_layer/ml/content_based_recommender_model/v1/model_artifacts/normalizer/",
    "sample_recommendations": "s3://yelpdataset-project/gold_layer/ml/content_based_recommender_model/v1/sample_recommendations/",
    "test_metrics": "s3://yelpdataset-project/gold_layer/ml/content_based_recommender_model/v1/evaluation/test_metrics/",
    "validation_results": "s3://yelpdataset-project/gold_layer/ml/content_based_recommender_model/v1/evaluation/validation_results/"
  },
  "build_completed_at_utc": "2026-08-01T18:29:11.864608+00:00",
  "build_s

## Automation handoff

After this notebook completes successfully:

1. The S3 model version is valid only when `model_manifest/` contains `"status": "READY"`.
2. A future `spark-submit` inference script should load:
   - `CountVectorizerModel` from `model_artifacts/category_vectorizer_model/`;
   - `Normalizer` from `model_artifacts/normalizer/`;
   - precomputed Parquet data from `business_feature_vectors/`;
   - weights from `model_configuration/`.
3. The inference script must **not** fit the vectorizer again.
4. When source data or feature logic changes, increment `MODEL_VERSION` and rerun this training notebook.
5. Required EMR role permissions are `s3:ListBucket`, `s3:GetObject`, `s3:PutObject`, and `s3:DeleteObject` for the configured input/output prefixes.

This notebook intentionally contains no AWS access keys or secrets.
